# VGGish + SVM

**Модель:** SVM (RBF) на VGGish-эмбеддингах (mean + std)

In [19]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import time
import torch
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, cross_val_predict
from sklearn.metrics import make_scorer, f1_score

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent.parent))

from shared import config, data_utils, train_utils
from shared.evaluate import find_optimal_threshold, evaluate
from shared.data_utils import build_feature_matrix
from shared.results_utils import save_result_csv

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(DEVICE)

cpu


In [20]:
vggish_model = torch.hub.load("harritaylor/torchvggish", "vggish")
vggish_model.eval()
vggish_model = vggish_model.to(DEVICE)

def extract_vggish(path: str) -> np.ndarray:
    with torch.no_grad():
        emb = vggish_model.forward(path).cpu().numpy()
    if emb.ndim == 1:
        emb = emb[np.newaxis, :]
    return np.concatenate([emb.mean(axis=0), emb.std(axis=0)]).astype(np.float32)

Using cache found in /Users/dk/.cache/torch/hub/harritaylor_torchvggish_master


In [21]:
(
    paths_trainval, labels_trainval, letters_trainval,
    paths_test, labels_test, letters_test,
) = data_utils.get_test_split()

print(f"Train+Val: {len(paths_trainval)} good: {np.sum(labels_trainval==0)}, bad: {np.sum(labels_trainval==1)}")
print(f"Test:      {len(paths_test)}  good: {np.sum(labels_test==0)},  bad: {np.sum(labels_test==1)}")

Train+Val: 2356 good: 1594, bad: 762
Test:      416  good: 281,  bad: 135


## Подбор гиперпараметров (GridSearchCV на train+val)

In [22]:
print("Извлечение эмбеддингов (train+val)...")
X_embeds_trainval = build_feature_matrix(paths_trainval, extract_vggish, n_jobs=1)
print("Извлечение эмбеддингов (test)...")
X_embeds_test     = build_feature_matrix(paths_test,     extract_vggish, n_jobs=1)

del vggish_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

X_trainval = np.hstack([X_embeds_trainval, letters_trainval])
X_test     = np.hstack([X_embeds_test,     letters_test])


Извлечение эмбеддингов (train+val)...
Извлечение эмбеддингов (test)...


In [23]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf", class_weight="balanced",
                probability=True, random_state=config.RANDOM_STATE)),
])
f1_bad_scorer = make_scorer(f1_score, pos_label=config.CLASS_BAD, average="binary")
param_grid = {
    "clf__C":     [0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 50.0],
    "clf__gamma": ["scale", "auto", 0.001, 0.01, 0.05, 0.1],
}

t0 = time.perf_counter()
grid = GridSearchCV(pipeline, param_grid, cv=5,
                    scoring=f1_bad_scorer, refit=True, n_jobs=-1)
grid.fit(X_trainval, labels_trainval)
train_time_sec = time.perf_counter() - t0

best_params = grid.best_params_
best_clf = grid.best_estimator_
print(f"Лучшие параметры: {best_params}")

Лучшие параметры: {'clf__C': 2.0, 'clf__gamma': 0.001}


## Оценка на test

In [24]:
y_proba = cross_val_predict(
    best_clf, X_trainval, labels_trainval, cv=5, method="predict_proba"
)[:, config.CLASS_BAD]
threshold = find_optimal_threshold(labels_trainval, y_proba)

y_proba_test = best_clf.predict_proba(X_test)[:, config.CLASS_BAD]
test_metrics = evaluate(labels_test, y_proba_test, threshold=threshold, verbose=True)
pd.DataFrame({
    "path":    paths_test,
    "y_true":  labels_test,
    "y_pred":  (y_proba_test >= threshold).astype(int),
    "y_proba": y_proba_test,
}).to_csv(exp_dir / "test_predictions.csv", index=False)

save_result_csv(
    exp_dir=exp_dir,
    experiment_id="exp_vggish_svm",
    experiment_name="VGGish + SVM (baseline)",
    accuracy=test_metrics["accuracy"],
    f1_macro=test_metrics["f1_macro"],
    f1_bad=test_metrics["f1_bad"],
    roc_auc=test_metrics["roc_auc"],
    precision_bad=test_metrics["precision_bad"],
    recall_bad=test_metrics["recall_bad"],
    threshold=test_metrics["threshold"],
    embed_dim=256,
    embed_dim_note="VGGish 128-dim * 2 (mean+std)",
    notes=f"GridSearchCV cv=5 + threshold | best_params={best_params} | thr={threshold:.2f}",
    train_time_sec=train_time_sec,
)

              precision    recall  f1-score   support

        good       0.83      0.79      0.81       281
         bad       0.60      0.65      0.63       135

    accuracy                           0.75       416
   macro avg       0.71      0.72      0.72       416
weighted avg       0.75      0.75      0.75       416

Threshold : 0.35
Accuracy  : 0.7476
F1-macro  : 0.7179
F1-bad    : 0.6263
ROC-AUC   : 0.7930


PosixPath('/Users/dk/Desktop/ВШЭ/ВКР/HSE_VKR_DetectingSpeechDefects/experiments/01_baselines/exp_vggish_svm/result.csv')